# Customer Retention Intelligence — Iteration 3
## Recommendations + Natural-Language Question Answering

This notebook is the **minimal final intelligence layer** for the project.

After running it top-to-bottom, you can ask questions such as:

- `Why is customer 7590-VHVEG likely to churn?`
- `What can make customer 7590-VHVEG stay?`
- `What is the churn probability for customer 7590-VHVEG?`
- `Why are customers generally churning?`

The answers are grounded in:
- the trained Logistic Regression model from Iteration 1
- SHAP explanations from Iteration 2
- the customer's actual feature values
- simple business recommendation rules

> Retention recommendations are suggestions based on observed model associations. They are not guaranteed causal effects.


## 0. Requirements

This notebook expects:

- Dataset: `../data/Customer_Churn_Dataset.xlsx`
- Model: `../models/baseline_retention_model.joblib`

If imports fail, run this once in a notebook cell:

```python
%pip install pandas numpy shap joblib matplotlib scikit-learn openpyxl
```


In [ ]:
import re
import pandas as pd
import numpy as np
import shap
import joblib

from IPython.display import display


## 1. Load model and dataset

In [ ]:
model = joblib.load("../models/baseline_retention_model.joblib")

df = pd.read_excel(
    "../data/Customer_Churn_Dataset.xlsx",
    sheet_name="01 Churn-Dataset"
)

print("Model loaded.")
print("Dataset shape:", df.shape)


## 2. Prepare the data

In [ ]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["TotalCharges"] = df["TotalCharges"].fillna(
    df["TotalCharges"].median()
)

customer_ids = df["customerID"].astype(str).copy()

df_model = df.drop(columns=["customerID"]).copy()

df_model["Churn"] = df_model["Churn"].replace({
    "No": 0,
    "Yes": 1
})

X = df_model.drop(columns=["Churn"])
y = df_model["Churn"]

print("Overall churn rate:", round(y.mean() * 100, 1), "%")


## 3. Recreate SHAP explanations

In [ ]:
preprocessor = model.named_steps["preprocessor"]
classifier = model.named_steps["classifier"]

X_processed = preprocessor.transform(X)
feature_names = preprocessor.get_feature_names_out()

explainer = shap.LinearExplainer(
    classifier,
    X_processed
)

shap_values = explainer(X_processed)

print("SHAP ready.")


## 4. Helpers to make technical feature names readable

In [ ]:
def clean_feature_name(name):
    """
    Convert scikit-learn feature names into easier business language.
    """
    name = str(name)

    if "__" in name:
        name = name.split("__", 1)[1]

    replacements = {
        "MonthlyCharges": "monthly charges",
        "TotalCharges": "total charges",
        "tenure": "tenure",
        "SeniorCitizen": "senior-citizen status",
        "numTechTickets": "number of technical-support tickets",
        "numAdminTickets": "number of administrative tickets",
        "PaperlessBilling": "paperless billing",
        "PaymentMethod": "payment method",
        "InternetService": "internet service",
        "OnlineSecurity": "online security",
        "OnlineBackup": "online backup",
        "DeviceProtection": "device protection",
        "TechSupport": "technical support",
        "StreamingTV": "streaming TV",
        "StreamingMovies": "streaming movies",
        "PhoneService": "phone service",
        "MultipleLines": "multiple lines",
        "Dependents": "dependents",
        "Partner": "partner status",
        "Contract": "contract type",
        "gender": "gender"
    }

    # Handle one-hot encoded categorical names, e.g. Contract_Month-to-month
    for raw, readable in replacements.items():
        prefix = raw + "_"
        if name.startswith(prefix):
            value = name[len(prefix):].replace("_", " ")
            return f"{readable} = {value}"

    return replacements.get(name, name.replace("_", " "))


## 5. Global churn-driver table

In [ ]:
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)

global_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": mean_abs_shap
})

global_importance["readable_feature"] = global_importance["feature"].apply(
    clean_feature_name
)

global_importance = global_importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

global_importance.head(15)


## 6. Customer lookup + explanation

In [ ]:
def find_customer_index(customer_id):
    matches = customer_ids[
        customer_ids.str.upper() == str(customer_id).upper()
    ].index

    if len(matches) == 0:
        return None

    return int(matches[0])


def explain_customer(customer_id, top_n=5):
    idx = find_customer_index(customer_id)

    if idx is None:
        return None

    row = X.iloc[[idx]]

    probability = float(model.predict_proba(row)[0][1])
    prediction = int(model.predict(row)[0])

    if probability >= 0.60:
        risk_level = "High"
    elif probability >= 0.30:
        risk_level = "Medium"
    else:
        risk_level = "Low"

    shap_df = pd.DataFrame({
        "feature": feature_names,
        "shap_value": shap_values.values[idx]
    })

    shap_df["readable_feature"] = shap_df["feature"].apply(
        clean_feature_name
    )

    shap_df["absolute_impact"] = shap_df["shap_value"].abs()

    shap_df = shap_df.sort_values(
        "absolute_impact",
        ascending=False
    )

    risk_factors = shap_df[
        shap_df["shap_value"] > 0
    ].head(top_n)

    protective_factors = shap_df[
        shap_df["shap_value"] < 0
    ].head(top_n)

    return {
        "index": idx,
        "customer_id": customer_ids.iloc[idx],
        "probability": probability,
        "prediction": prediction,
        "risk_level": risk_level,
        "risk_factors": risk_factors[
            ["readable_feature", "shap_value"]
        ].to_dict("records"),
        "protective_factors": protective_factors[
            ["readable_feature", "shap_value"]
        ].to_dict("records")
    }


## 7. Retention recommendation engine

The rules below use the customer's actual data to suggest reasonable actions.

They intentionally say **"consider"** rather than claiming an action will definitely prevent churn.


In [ ]:
def retention_recommendations(customer_id):
    idx = find_customer_index(customer_id)

    if idx is None:
        return []

    c = df.iloc[idx]
    recommendations = []

    # Contract
    if str(c.get("Contract", "")).lower() == "month-to-month":
        recommendations.append(
            "Consider offering an incentive for a longer-term contract, "
            "because the customer is currently on a month-to-month plan."
        )

    # Tenure
    if float(c.get("tenure", 999)) <= 12:
        recommendations.append(
            "Consider stronger onboarding or early-life engagement, "
            "because this customer has relatively short tenure."
        )

    # Monthly charges
    monthly = float(c.get("MonthlyCharges", 0))
    dataset_median_monthly = float(df["MonthlyCharges"].median())

    if monthly > dataset_median_monthly:
        recommendations.append(
            "Consider a plan review, bundle optimization, or targeted pricing offer, "
            "because this customer's monthly charges are above the dataset median."
        )

    # Tech support
    if str(c.get("TechSupport", "")).lower() == "no":
        recommendations.append(
            "Consider proactive technical-support outreach or a support package."
        )

    # Online security
    if str(c.get("OnlineSecurity", "")).lower() == "no":
        recommendations.append(
            "Consider educating the customer about available security features "
            "or offering an appropriate security bundle."
        )

    # Tickets
    tech_tickets = float(c.get("numTechTickets", 0))
    if tech_tickets >= 2:
        recommendations.append(
            "Prioritize service-recovery outreach because this customer has multiple technical-support tickets."
        )

    admin_tickets = float(c.get("numAdminTickets", 0))
    if admin_tickets >= 2:
        recommendations.append(
            "Review unresolved billing or administrative friction because this customer has multiple administrative tickets."
        )

    # Payment
    payment = str(c.get("PaymentMethod", "")).lower()
    if "electronic check" in payment:
        recommendations.append(
            "Consider encouraging a simpler automatic-payment method if appropriate."
        )

    # Keep output focused
    if not recommendations:
        recommendations.append(
            "No obvious rule-based intervention was identified. Review the customer's SHAP risk factors before taking action."
        )

    return recommendations[:5]


## 8. Turn customer explanation into plain English

In [ ]:
def customer_explanation_text(customer_id):
    info = explain_customer(customer_id)

    if info is None:
        return f"Customer {customer_id} was not found."

    lines = [
        f"Customer {info['customer_id']} has a predicted churn probability of "
        f"{info['probability']:.1%}, which places them in the {info['risk_level'].lower()}-risk group."
    ]

    if info["risk_factors"]:
        lines.append("\nMain factors pushing the model toward churn:")
        for i, item in enumerate(info["risk_factors"], start=1):
            lines.append(
                f"{i}. {item['readable_feature']}"
            )

    if info["protective_factors"]:
        lines.append("\nMain factors pushing the model toward staying:")
        for i, item in enumerate(info["protective_factors"], start=1):
            lines.append(
                f"{i}. {item['readable_feature']}"
            )

    return "\n".join(lines)


## 9. Turn retention recommendations into plain English

In [ ]:
def retention_text(customer_id):
    info = explain_customer(customer_id)

    if info is None:
        return f"Customer {customer_id} was not found."

    actions = retention_recommendations(customer_id)

    lines = [
        f"Possible retention actions to consider for customer {info['customer_id']}:"
    ]

    for i, action in enumerate(actions, start=1):
        lines.append(f"{i}. {action}")

    lines.append(
        "\nThese are business recommendations based on model associations "
        "and customer attributes; they are not guaranteed causal effects."
    )

    return "\n".join(lines)


## 10. General churn explanation

In [ ]:
def overall_churn_text(top_n=8):
    top = global_importance.head(top_n)

    lines = [
        f"The observed churn rate in this dataset is {y.mean():.1%}.",
        "",
        "The model's strongest overall churn-related features are:"
    ]

    for i, row in top.iterrows():
        lines.append(
            f"{i + 1}. {row['readable_feature']}"
        )

    lines.append(
        "\nThese are predictive associations learned by the model. "
        "They should not automatically be interpreted as causal reasons."
    )

    return "\n".join(lines)


## 11. Natural-language question interface

This is the function you will use.

Supported questions include:

- Why is customer `<ID>` likely to churn?
- What can make customer `<ID>` stay?
- What is the churn probability for customer `<ID>`?
- Why are customers generally churning?


In [ ]:
def extract_customer_id(question):
    # First try the normal IBM-style customer ID pattern.
    match = re.search(
        r"\b[A-Za-z0-9]{4}-[A-Za-z0-9]{5}\b",
        question
    )

    if match:
        return match.group(0)

    # Fallback: check whether any known customer ID appears in the question.
    q_upper = question.upper()

    for cid in customer_ids:
        if cid.upper() in q_upper:
            return cid

    return None


def ask(question):
    q = question.strip().lower()
    customer_id = extract_customer_id(question)

    # General churn questions
    general_terms = [
        "generally",
        "overall",
        "in general",
        "why are customers",
        "why customers"
    ]

    if any(term in q for term in general_terms) and "churn" in q:
        return overall_churn_text()

    # Customer-specific questions
    if customer_id is not None:

        if (
            "stay" in q
            or "retain" in q
            or "retention" in q
            or "what can" in q
            or "what should" in q
        ):
            return retention_text(customer_id)

        if (
            "probability" in q
            or "risk" in q
            or "chance" in q
        ):
            info = explain_customer(customer_id)

            if info is None:
                return f"Customer {customer_id} was not found."

            return (
                f"Customer {info['customer_id']} has a predicted churn probability "
                f"of {info['probability']:.1%} and is classified as "
                f"{info['risk_level']} risk."
            )

        if (
            "why" in q
            or "churn" in q
            or "leave" in q
        ):
            return customer_explanation_text(customer_id)

    return (
        "I can currently answer questions such as:\\n"
        "- Why is customer <ID> likely to churn?\\n"
        "- What can make customer <ID> stay?\\n"
        "- What is the churn probability for customer <ID>?\\n"
        "- Why are customers generally churning?"
    )


## 12. Test with a real customer from the dataset

In [ ]:
example_customer_id = customer_ids.iloc[0]

print("Example customer ID:", example_customer_id)


In [ ]:
print(
    ask(f"Why is customer {example_customer_id} likely to churn?")
)


In [ ]:
print(
    ask(f"What can make customer {example_customer_id} stay?")
)


In [ ]:
print(
    ask(f"What is the churn probability for customer {example_customer_id}?")
)


In [ ]:
print(
    ask("Why are customers generally churning?")
)


# Use it yourself

Change the text below to any supported question and rerun the cell.


In [ ]:
QUESTION = f"Why is customer {example_customer_id} likely to churn?"

print("Question:")
print(QUESTION)

print("\nAnswer:")
print(ask(QUESTION))


# Iteration 3 complete

You now have a working question-answering layer that can answer:

- why a particular customer is predicted to churn
- which factors push that customer toward staying
- what retention actions may be worth considering
- what the customer's churn probability is
- which factors are generally most important to churn predictions

The next optional step is only **UI**: putting `ask()` behind a small Streamlit chat interface so you can type questions into a web app instead of editing a notebook cell.
